# Importing Dependencies


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as Functional
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as Transform
from torchvision.models import vgg19,VGG19_Weights
from torch.cuda.amp import GradScaler , autocast
from torchvision.utils import save_image,make_grid

import os 
import copy
import numpy as np  
import PIL
from PIL import Image 
from torchsummary import summary

import matplotlib.pyplot as plt
from tqdm import tqdm
import random

# Device Information and Seeding

In [ ]:
#setting random seed for reproduciblity 
torch.manual_seed( 42 )
# Check for GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using device: {device}")

# Preparing Directories path

In [ ]:
train_person_images_path="D:\\pytorch_projects\\U_net_(1)\\Dataset\\Train\\image"
train_agnostic_images_path="D:\\pytorch_projects\\U_net_(1)\\Dataset\\Train\\agnostic-v3.2"

test_person_images_path="D:\\pytorch_projects\\U_net_(1)\\Dataset\\Test\\image"
test_agnostic_images_path="D:\\pytorch_projects\\U_net_(1)\\Dataset\\Test\\agnostic-v3.2"

train_directories=[train_person_images_path,train_agnostic_images_path]
test_directories =[test_person_images_path,test_agnostic_images_path]

# Creating UNet DataSet

In [ ]:
# Custom UNet dataset for training 
class U_net_dataset(Dataset):
    def __init__(self, Directories,Height,Width): 
        
        #validation of heigth and width is integer or not 
        if not isinstance(Height,int) or not isinstance(Width,int):
            raise TypeError(f"Height and width must be integers. Got: height={type(Height)}, width={type(Width)}")
        
        self.directories = Directories  # List of directories for images
        self.height=Height
        self.width=Width
     
        # Load and sort file paths
        self.person_images   = sorted([os.path.join(self.directories[0], f) for f in os.listdir(self.directories[0])])     
        self.agnostic_images = sorted([os.path.join(self.directories[1], f) for f in os.listdir(self.directories[1])])     
         
                      
        # Transformation [Resize, To Tensor with normalization to 0-1]
        self.transform = Transform.Compose([
            Transform.Resize((self.height, self.width),interpolation=Image.LANCZOS),
            Transform.ToTensor()  # Converts PIL Image to tensor with shape [C, H, W] and values [0,1]
        ])
        
        if not  len(self.person_images) == len(self.agnostic_images):
            raise ValueError("Mismatch in dataset image counts!")
            
            
        
    # Add type annotation for clarity (optional but helps IDE)
    def __getitem__(self, idx: int, visualization: bool = False) -> tuple:
        # Load images as PIL Images
        person_img   = Image.open(self.person_images[idx]).convert("RGB")
        agnostic_img = Image.open(self.agnostic_images[idx]).convert("RGB")
        

        # Transform PIL images to PyTorch tensors
        person_tensor   = self.transform(person_img)    # [3, H, W]
        agnostic_tensor = self.transform(agnostic_img) # [3, H, W]
        
        return person_tensor, agnostic_tensor
        

    def __len__(self):
        return len(self.person_images)
      
            

# creating the instances of UNet dataset class  for training and the height and width =224 for taining  
train_dataset = U_net_dataset(train_directories,224,224)
test_dataset= U_net_dataset(test_directories,224,224)

idx = random.randint(0,11400) #getting an random index number 
person_tensor, agnostic_tensor = train_dataset.__getitem__(idx)

print(f"Person Tensor shape:   {person_tensor.shape}, dtype={person_tensor.dtype}")
print(f"Agnostic Tensor shape: {agnostic_tensor.shape}, dtype={agnostic_tensor.dtype}")


# Visualization: permute channels to HWC for plt.imshow()
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(person_tensor.permute(1, 2, 0).numpy())
axes[0].set_title("Person Image")
axes[0].axis("off")

axes[1].imshow(agnostic_tensor.permute(1, 2, 0).numpy())
axes[1].set_title("Agnostic Image")
axes[1].axis("off")

plt.tight_layout()
plt.show()


# Model Architecture

In [ ]:

#---------------------------------------- Functions --------------------------------------------#

# Double Convolution Block with Batch Normalization (*)
def double_conv(in_channels, out_channels): 
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(inplace=True),
        nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(inplace=True)
    )


# DownSample function
# DownSample function: double conv followed by strided conv for downsampling
def down_sample(in_channels, out_channels):
    conv = double_conv(in_channels, out_channels)
    pool = nn.Conv2d(out_channels, out_channels, kernel_size=2, stride=2)
    return conv,pool


# UpSample function
def up_sample(in_channels, out_channels):
    up = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)
    conv = double_conv(in_channels, out_channels)
    return up, conv

#------------------------------------------- UNet Class ----------------------------------------#

class UNet(nn.Module):

    # Constructor:
    def __init__(self, in_channels, out_channels):
        super().__init__()

        # Downsampling path (encoder)
        self.down1_conv, self.down1_pool = down_sample(in_channels, 64)
        self.down2_conv, self.down2_pool = down_sample(64, 128)
        self.down3_conv, self.down3_pool = down_sample(128, 256)
        self.down4_conv, self.down4_pool = down_sample(256, 512)

        # Bottleneck layer
        self.bottleneck = double_conv(512, 1024)

        # Upsampling path (decoder)
        self.up1_up, self.up1_conv = up_sample(1024, 512)
        self.up2_up, self.up2_conv = up_sample(512, 256)
        self.up3_up, self.up3_conv = up_sample(256, 128)
        self.up4_up, self.up4_conv = up_sample(128, 64)

        # Final output layer: Converts feature maps to segmentation output
        self.out_conv = nn.Conv2d(64, out_channels, kernel_size=1)

    # Forward Pass
    def forward(self, input_tensor):

        #------------------------------------ Downsampling -------------------------------------#   
        down1 = self.down1_conv(input_tensor)
        p1 = self.down1_pool(down1)

        down2 = self.down2_conv(p1)
        p2 = self.down2_pool(down2)

        down3 = self.down3_conv(p2)
        p3 = self.down3_pool(down3)

        down4 = self.down4_conv(p3)
        p4 = self.down4_pool(down4)
        

        #------------------------------------ Bottleneck ----------------------------------------#      
        bottleneck = self.bottleneck(p4)
       

        #------------------------------------- Upsampling ---------------------------------------#    
        up1 = self.up1_up(bottleneck)
        up1 = torch.cat([up1, down4], dim=1)
        up1 = self.up1_conv(up1)

        up2 = self.up2_up(up1)
        up2 = torch.cat([up2, down3], dim=1)
        up2 = self.up2_conv(up2)

        up3 = self.up3_up(up2)
        up3 = torch.cat([up3, down2], dim=1)
        up3 = self.up3_conv(up3)

        up4 = self.up4_up(up3)
        up4 = torch.cat([up4, down1], dim=1)
        up4 = self.up4_conv(up4)
       

        #------------------------------------ Output Layer --------------------------------------#
        out = self.out_conv(up4)
        return out
    
    
   

# Model Summary(Total tainable prams = 32,431,363)

In [ ]:
model = UNet(in_channels=3, out_channels=3).to(device)
summary(model, input_size=(3, 224, 224))

# Weight Initalizer

In [ ]:
# -------------------------- Weight Initialization Class -------------------------------
class WeightInitializer:
    def __init__(self, mode="he"):
        self.mode = mode

    def __call__(self, module):
        if isinstance(module, (nn.Conv2d, nn.ConvTranspose2d)):
            if self.mode == "he":
                nn.init.kaiming_normal_(module.weight)
            elif self.mode == "xavier":
                nn.init.xavier_normal_(module.weight)
            else:
                raise ValueError(f"Unsupported init mode: {self.mode}")

            if module.bias is not None:
                nn.init.constant_(module.bias, 0)

        elif isinstance(module, nn.BatchNorm2d):
            nn.init.constant_(module.weight, 1)
            nn.init.constant_(module.bias, 0)

# Loss Functions 

In [ ]:
# ---------------------- Perceptual Loss using pretrained VGG19 ------------------------
class VGGPerceptualLoss(nn.Module):
    def __init__(self):
        super().__init__()
        vgg = vgg19(weights=VGG19_Weights.DEFAULT)
        vgg_layers = list(vgg.features.children())[:16] # Up to conv3_3
        self.slice = nn.Sequential(*vgg_layers)  

        for param in self.slice.parameters():
            param.requires_grad = False
        self.slice.eval()

    def forward(self, input, target):
        input = (input + 1) / 2  # Normalize from [-1, 1] to [0, 1]
        target = (target + 1) / 2
        input_features = self.slice(input)
        target_features = self.slice(target)
        return Functional.l1_loss(input_features, target_features)  #returns the feature loss of the both tensor
    
perceptual_model=VGGPerceptualLoss().to(device) #creating instances and moving to gpu


# ---------------------- Perceptual Loss + L1 Loss ------------------------
def combined_loss(output, target, perceptual_model,  alpha, beta):
    pixel_loss = Functional.l1_loss(output, target)
    perceptual_loss = perceptual_model(output, target)
    return alpha * pixel_loss + beta * perceptual_loss 

# Accuracy Metrics 


In [ ]:
# ---------------------- SSIM ------------------------
class SSIM(nn.Module):
    def __init__(self, window_size=11, sigma=1.5, data_range=1.0):
        super(SSIM, self).__init__()
        self.window_size = window_size
        self.sigma = sigma
        self.data_range = data_range
        self.C1 = (0.01 * data_range) ** 2
        self.C2 = (0.03 * data_range) ** 2
        self.window = None

    def create_gaussian_kernel(self, window_size, sigma, channel, device, dtype):
        coords = torch.arange(window_size, device=device, dtype=dtype) - window_size // 2
        gaussian_1d = torch.exp(-(coords ** 2) / (2 * sigma ** 2))
        gaussian_1d /= gaussian_1d.sum()
        kernel_2d = torch.matmul(gaussian_1d.unsqueeze(1), gaussian_1d.unsqueeze(0))
        kernel_2d = kernel_2d.expand(channel, 1, window_size, window_size).contiguous()
        return kernel_2d

    def forward(self, tensor1, tensor2):
        device = tensor1.device
        dtype = tensor1.dtype
        channels = tensor1.size(1)

        # Create or recreate Gaussian kernel if necessary (always recreate for safety here)
        self.window = self.create_gaussian_kernel(self.window_size, self.sigma, channels, device, dtype)

        # Explicitly cast tensors to same dtype (to avoid dtype mismatch errors)
        tensor1 = tensor1.to(dtype)
        tensor2 = tensor2.to(dtype)
        self.window = self.window.to(dtype)

        # Calculate local means
        mu1 = Functional.conv2d(tensor1, self.window, padding=self.window_size // 2, groups=channels)
        mu2 = Functional.conv2d(tensor2, self.window, padding=self.window_size // 2, groups=channels)

        mu1_sq = mu1.pow(2)
        mu2_sq = mu2.pow(2)
        mu1_mu2 = mu1 * mu2

        sigma1_sq = Functional.conv2d(tensor1 * tensor1, self.window, padding=self.window_size // 2, groups=channels) - mu1_sq
        sigma2_sq = Functional.conv2d(tensor2 * tensor2, self.window, padding=self.window_size // 2, groups=channels) - mu2_sq
        sigma12 = Functional.conv2d(tensor1 * tensor2, self.window, padding=self.window_size // 2, groups=channels) - mu1_mu2

        # Make constants tensors on correct device and dtype
        C1 = torch.tensor(self.C1, device=device, dtype=dtype)
        C2 = torch.tensor(self.C2, device=device, dtype=dtype)

        numerator1 = 2 * mu1_mu2 + C1
        numerator2 = 2 * sigma12 + C2
        denominator1 = mu1_sq + mu2_sq + C1
        denominator2 = sigma1_sq + sigma2_sq + C2

        ssim_map = (numerator1 * numerator2) / (denominator1 * denominator2)

        # Average over batch, channel, height, width -> scalar
        ssim_scalar = ssim_map.mean(dim=(0, 1, 2, 3))

        return ssim_scalar

 
    
# --------------------- Pixel Accuracy with SSIM Measuring Function ------------------------------
def combined_accuracy(outputs, targets,SSIM,p5_threshold=0.05,p10_threshold=0.1):
    diff = torch.abs(outputs - targets)  # [B, C, H, W]
    p5_correct = (diff < p5_threshold).all(dim=1)  # All channels within threshold 5%
    p10_correct= (diff<p10_threshold).all(dim=1)
    SSIM_score=SSIM(outputs,targets)
    return p5_correct.float().mean(),p10_correct.float().mean(),SSIM_score



# Data Loaders

In [ ]:
train_loader = DataLoader(train_dataset,batch_size=16,shuffle=True)
test_loader = DataLoader(test_dataset,batch_size=16 , shuffle= False)

print(f"Number of training batches: {len(train_loader)}")
print(f"Number of testing batches: {len(test_loader)}")


for batch in train_loader:
    input_tensor, target_tensor = batch
    print(f"input_tensor batch shape: {input_tensor.shape} \ntarget_tensor batch shape: {target_tensor.shape}")
    break

# Instanciating All The Classes

In [ ]:
model = UNet(in_channels=3, out_channels=3).to(device) #creating instance of the Unet model
initializer = WeightInitializer(mode="he")             # creating instance of initializer
model.apply(initializer)                               # Apply weight initialization

perceptual_loss_model = VGGPerceptualLoss().to(device) #creating instance of perpetual loss model 
SSIM_Metric = SSIM().to(device)                         #creating instance of SSIM metric

# Creating a sample Input and Ground truth for Visualization on each epoch

In [ ]:
# Define paths
person_image_path = "D:\\pytorch_projects\\U_net_(1)\\Image_Gen\\Input\\person.jpg"
ground_truth_image_path = "D:\\pytorch_projects\\U_net_(1)\\Image_Gen\\Ground_truth\\ground_truth.jpg"

# Load images
person_img = Image.open(person_image_path).convert("RGB")

ground_truth_img = Image.open(ground_truth_image_path).convert("RGB")
ground_truth_img = ground_truth_img.resize((224, 224), Image.LANCZOS)

# Save the resized ground truth image
ground_truth_save_path = "D:\\pytorch_projects\\U_net_(1)\\Image_Gen\\Ground_truth\\224_version\\ground_truth.jpg"
os.makedirs(os.path.dirname(ground_truth_save_path), exist_ok=True)
ground_truth_img.save(ground_truth_save_path)


# Define transformation: Resize to 224x224 and convert to tensor
transform = Transform.Compose([
    Transform.Resize((224, 224)),
    Transform.ToTensor()  # Converts to [C, H, W] and scales to [0,1]
])

# Apply transformations
person_tensor = transform(person_img)          # [3, 224, 224]           # [3, 224, 224]
ground_truth_tensor = transform(ground_truth_img)  # [3, 224, 224]

# Add batch dimension
input_validation_tensor = person_tensor.unsqueeze(0).to(device)      #[1,6,224,224]
print(input_validation_tensor.shape)

# Training Configuration

In [ ]:

# Hyperparameters
EPOCHS = 150
LEARNING_RATE = 1e-4 #after 60 epoch learning rate =1e-5
ALPHA = 1
BETA = 1
ALPHA_DECAY = 0.95
BETA_GROWTH = 1
patience = 5  #after 60 epoch patience =5
min_delta = 0.001

best_model_state=None

#optimizer ,loss, accuracy metric
optimizer=optim.Adam(model.parameters(),lr=LEARNING_RATE)
""" 
->Loss Function =   combined_loss()
->Accuracy Metric = combined_accuracy()
"""

# Generated image  path 
per_epoch_img_gen_path = "D:\\pytorch_projects\\U_net_(1)\\Image_Gen\\Generated"
os.makedirs(per_epoch_img_gen_path, exist_ok=True)

# Checkpoint path 
check_point_path = "D:\\pytorch_projects\\U_net_(1)\\Checkpoints(UNet[1])"
os.makedirs(check_point_path, exist_ok=True)

#Best model path
Best_model_path=  "D:\\pytorch_projects\\U_net_(1)\\Checkpoints(UNet[1])\\Best_model"
os.makedirs(Best_model_path, exist_ok=True)

scaler=GradScaler()



# Validating Checkpoints

In [ ]:
# ----------- Load checkpoint if exists --------------
latest_checkpoint_file = os.path.join(check_point_path, "latest_checkpoint.pth")

if os.path.exists(latest_checkpoint_file):
    print(f"Loading checkpoint from {latest_checkpoint_file} ...")
    checkpoint = torch.load(latest_checkpoint_file, map_location=device)
    
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scaler.load_state_dict(checkpoint['scaler_state_dict'])
    
    start_epoch = checkpoint['epoch']
    ALPHA = checkpoint['alpha']
    BETA = checkpoint['beta']
    best_loss = checkpoint['best_loss']
    no_improve_count = checkpoint['no_improve_count']
    
    train_loss_history = checkpoint.get('train_loss_history', [])
    train_p5_acc_history = checkpoint.get('train_p5_acc_history', [])
    train_p10_acc_history = checkpoint.get('train_p10_acc_history', [])
    train_SSIM_score_history = checkpoint.get('train_SSIM_score_history', [])
    
    print(f"Checkpoint loaded. Resuming training from epoch {start_epoch}.")
    
    
else:
    print("No checkpoint found. Starting training from scratch.")
    start_epoch = 0
    ALPHA = 1.0  # initial alpha
    BETA = 1  # initial beta
    best_loss = float('inf')
    no_improve_count = 0
    train_loss_history=[]
    train_p5_acc_history = []
    train_p10_acc_history = []
    train_SSIM_score_history = []


# Training Loop

In [ ]:

# ----------- Training loop --------------------------
for epoch in range(start_epoch, EPOCHS):
    model.train()
    epoch_loss = 0.0
    epoch_p5_acc = 0.0
    epoch_p10_acc = 0.0
    epoch_SSIM_score = 0.0
    total_samples = 0

    loop = tqdm(train_loader, desc=f"Epoch [{epoch + 1}/{EPOCHS}]", leave=False)

    for input_tensor, agnostic_tensor in loop:
        input_tensor, agnostic_tensor= input_tensor.to(device), agnostic_tensor.to(device)
        batch_size = input_tensor.size(0)
        total_samples += batch_size

        optimizer.zero_grad()

        with torch.cuda.amp.autocast(enabled=True):
            outputs = model(input_tensor)
            loss = combined_loss(outputs, agnostic_tensor, perceptual_loss_model, ALPHA, BETA)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        p5_acc, p10_acc, ssim_score = combined_accuracy(outputs, agnostic_tensor, SSIM_Metric)

        epoch_loss += loss.item() * batch_size
        epoch_p5_acc += p5_acc.item() * batch_size
        epoch_p10_acc += p10_acc.item() * batch_size
        epoch_SSIM_score += ssim_score.item() * batch_size

        loop.set_postfix({
            "Combined Loss": f"{loss.item():.4f}",
            "5% Pixel Acc": f"{p5_acc.item() * 100:.2f}%",
            "10% Pixel Acc": f"{p10_acc.item() * 100:.2f}%",
            "SSIM score": f"{ssim_score.item():.4f}",
        })

#---------------- Avg Losses per epoch----------------- 
    avg_loss = epoch_loss / total_samples
    avg_p5_acc = epoch_p5_acc / total_samples
    avg_p10_acc = epoch_p10_acc / total_samples
    avg_ssim_score = epoch_SSIM_score / total_samples

    train_loss_history.append(avg_loss)
    train_p5_acc_history.append(avg_p5_acc)
    train_p10_acc_history.append(avg_p10_acc)
    train_SSIM_score_history.append(avg_ssim_score)

    print(f"[Epoch {epoch+1}/{EPOCHS}] Avg Loss: {avg_loss:.4f}, 5% Pixel Acc: {avg_p5_acc * 100:.2f}%, "
          f"10% Pixel Acc: {avg_p10_acc * 100:.2f}%, SSIM score: {avg_ssim_score:.4f}")

    # Save the output of the genereated image per eopch 
    model.eval()
    with torch.no_grad():
        sample_input = input_validation_tensor
        sample_input = sample_input.to(device)
        sample_output = model(sample_input)
        save_image(sample_output, os.path.join(per_epoch_img_gen_path, f"epoch_{epoch+1}.png"))
        
        
        #showing the generated sample image 
        output_img_grid = make_grid(sample_output.cpu(), normalize=True)
        output_img_np = output_img_grid.permute(1, 2, 0).numpy()  # CHW -> HWC for matplotlib
        
        # Plot side by side
        fig, axes = plt.subplots(1, 2, figsize=(12, 6))

       # Show generated image
        axes[0].imshow(output_img_np)
        axes[0].set_title(f"Generated Output - Epoch {epoch+1}")
        axes[0].axis('off')

       # Show ground truth image
        axes[1].imshow(ground_truth_img)
        axes[1].set_title("Ground Truth")
        axes[1].axis('off')
        plt.show()


    # Prepare checkpoint dict
    checkpoint = {
        'epoch': epoch + 1,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scaler_state_dict': scaler.state_dict(),
        'alpha': ALPHA,
        'beta': BETA,
        'best_loss': best_loss,
        'no_improve_count': no_improve_count,
        'train_loss_history': train_loss_history,
        'train_p5_acc_history': train_p5_acc_history,
        'train_p10_acc_history': train_p10_acc_history,
        'train_SSIM_score_history': train_SSIM_score_history,
        
        
    }

    # Save checkpoint for this epoch
    checkpoint_save_path = os.path.join(check_point_path, f"Checkpoint_{epoch+1}.pth")
    torch.save(checkpoint, checkpoint_save_path)

    # Save latest checkpoint (overwrite)
    latest_checkpoint_file = os.path.join(check_point_path, "latest_checkpoint.pth")
    torch.save(checkpoint, latest_checkpoint_file)

    # Update ALPHA and BETA for next epoch
    ALPHA *= ALPHA_DECAY  # 5% decrease per epoch 
    BETA *= BETA_GROWTH   # Remains constant

    # ------------------ Early Stopping Logic ------------------
    if best_loss == float('inf') or ((best_loss - avg_loss) / best_loss) > min_delta:
        best_loss = avg_loss
        no_improve_count = 0
        best_model_state = copy.deepcopy(model.state_dict())
        torch.save(best_model_state, os.path.join(Best_model_path, "best_model.pth"))  #if early stopping is not triggerd then This will be the best model overwriting the best model each epoch 
        print("Model improved and saved.")
    else:
        no_improve_count += 1
        print(f"No improvement in loss for {no_improve_count} epoch(s).")

        if no_improve_count >= patience:
            print("Early stopping triggered. Rolling back to best model weights...")
            if best_model_state is not None:
                model.load_state_dict(best_model_state)
                model.to(device)
                torch.save(best_model_state, os.path.join(Best_model_path, "best_model.pth"))#Over-write the Current best model and save at the same location 
            else:
                print("Warning: Best model cannot be saved  !!.")
            break

# Loss - Accuracy Curves

In [ ]:
# Path to the latest checkpoint file 
latest_checkpoint_file = os.path.join(check_point_path, "latest_checkpoint.pth")

# Load checkpoint
checkpoint = torch.load(latest_checkpoint_file, map_location=device)

# Extract training histories from checkpoint
train_loss_history = checkpoint.get('train_loss_history', [])
train_p5_acc_history = checkpoint.get('train_p5_acc_history', [])
train_p10_acc_history = checkpoint.get('train_p10_acc_history', [])
train_SSIM_score_history = checkpoint.get('train_SSIM_score_history', [])

def plot_training_metrics(loss_history,p5_acc_history,p10_acc_history, ssim_history):
    
    epochs = range(1, len(loss_history) + 1)
    
    plt.figure(figsize=(12, 8))
    
    # Plot Loss
    plt.subplot(2, 2, 1)
    plt.plot(epochs, loss_history, 'r-', label='Train Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Training Loss over Epochs')
    plt.grid(True)
    plt.legend()
    
    # Plot 5% Pixel Accuracy
    plt.subplot(2, 2, 2)
    plt.plot(epochs, [acc * 100 for acc in p5_acc_history], 'b-', label='5% Pixel Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy (%)')
    plt.title('5% Pixel Accuracy over Epochs')
    plt.grid(True)
    plt.legend()
    
    # Plot 10% Pixel Accuracy
    plt.subplot(2, 2, 3)
    plt.plot(epochs, [acc * 100 for acc in p10_acc_history], 'g-', label='10% Pixel Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy (%)')
    plt.title('10% Pixel Accuracy over Epochs')
    plt.grid(True)
    plt.legend()
    
    # Plot SSIM Score
    plt.subplot(2, 2, 4)
    plt.plot(epochs, ssim_history, 'm-', label='SSIM Score')
    plt.xlabel('Epoch')
    plt.ylabel('SSIM')
    plt.title('SSIM Score over Epochs')
    plt.grid(True)
    plt.legend()
    
    plt.tight_layout()
    plt.show()
    
    
# Plot from loaded histories
plot_training_metrics(train_loss_history,train_p5_acc_history,train_p10_acc_history,train_SSIM_score_history)

# Evaluation

# Inferrence

In [ ]:

# --- Set device ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- Load the best model ---
model = UNet(in_channels=3, out_channels=3).to(device)
best_model_path = os.path.join(Best_model_path, "best_model.pth")
model.load_state_dict(torch.load(best_model_path, map_location=device))
model.eval()
print("(*) Best model loaded successfully.")

# --- Create test dataset and loader ---
test_dataset = U_net_dataset(test_directories,224,224)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)
print(f"Total test batches: {len(test_loader)}")

# --- Set parameters ---
batch_index =20
ALPHA = 1
BETA = 1

# --- Get the specific batch ---
for i, (inputs, targets) in enumerate(test_loader):
    if i == batch_index:
        print(f"\n\nFound batch [{batch_index}]\n\n")
        inputs = inputs.to(device)
        targets = targets.to(device)
        person_tensor = targets  # Ensuring person_tensor is on the same device
        break

# --- Run inference on the selected batch ---
with torch.no_grad():
    outputs = model(inputs)

    # Evaluate metrics
    p5_acc, p10_acc, ssim_score = combined_accuracy(outputs, targets, SSIM_Metric)
    loss = combined_loss(outputs, person_tensor, perceptual_loss_model, ALPHA, BETA)

    print(f"\n(*) Evaluation Metrics for Batch {batch_index}:")
    print(f"Average Loss:        {loss.item() /16:.4f}")
    print(f"5% Pixel Accuracy:   {p5_acc.item() * 100:.2f}%")
    print(f"10% Pixel Accuracy:  {p10_acc.item() * 100:.2f}%")
    print(f"SSIM Score:          {ssim_score.item():.4f}")

    # --- Visualize Image ---
    batch_size = inputs.size(0)
    for idx in range(batch_size):
        target_np = targets[idx].permute(1, 2, 0).cpu().numpy().clip(0, 1)
        output_np = outputs[idx].permute(1, 2, 0).cpu().numpy().clip(0, 1)

        fig, axes = plt.subplots(1, 2, figsize=(8, 4))
        axes[0].imshow(target_np)
        axes[0].set_title(f"Ground Truth {idx}")
        axes[0].axis("off")

        axes[1].imshow(output_np)
        axes[1].set_title(f"Generated Output {idx}")
        axes[1].axis("off")

        plt.tight_layout()
        plt.show()
